# 04 — Weekly Team Strength

This notebook updates team strength sequentially using the new in season evidence created in `03_Inseason_Feature_Engineering.ipynb`.

The original preseason model remains unchanged.

The weekly model follows a sequential principle:

> The previous week's team strength is the prior. The most recently completed week provides new evidence.

For each target week, this notebook:

- Loads the frozen preseason ratings for reference
- Loads the previous week's team strength state
- Loads the target week's new in season feature table
- Applies the opponent adjusted weekly strength update
- Preserves separate offensive and defensive weekly signals
- Calculates current weekly rankings
- Compares rank and strength movement to the previous week
- Saves a separate weekly team strength file

No original preseason file or prior weekly file is overwritten.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


## Paths and Weekly Settings

This notebook expects the output from:

`03_Inseason_Feature_Engineering.ipynb`

For Week 3:

- `TARGET_WEEK = 3`
- `PRIOR_WEEK = 2`

The Week 2 team strength file becomes the entering prior for Week 3.

In [2]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

SEASON = 2026
TARGET_WEEK = 9
PRIOR_WEEK = TARGET_WEEK - 1

PRESEASON_STRENGTH_PATH = (
    PROCESSED_DIR
    / "2026_team_strength.parquet"
)

PRIOR_WEEK_STRENGTH_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{PRIOR_WEEK:02d}_sequential_team_strength.parquet"
)

INSEASON_FEATURES_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_inseason_features.parquet"
)

OUTPUT_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_sequential_team_strength.parquet"
)

print("Preseason ratings:", PRESEASON_STRENGTH_PATH)
print("Prior-week ratings:", PRIOR_WEEK_STRENGTH_PATH)
print("New in-season features:", INSEASON_FEATURES_PATH)
print("Weekly output:", OUTPUT_PATH)

Preseason ratings: ..\..\data\processed\2026_team_strength.parquet
Prior-week ratings: ..\..\data\processed\weekly\week_08_sequential_team_strength.parquet
New in-season features: ..\..\data\processed\weekly\week_09_inseason_features.parquet
Weekly output: ..\..\data\processed\weekly\week_09_sequential_team_strength.parquet


# Load Frozen Preseason Team Strength

The preseason team strength file is the final output of the original `05_Team_Strength_Model.ipynb`.

Its `team_strength` value is already expressed in point differential terms, so it can serve directly as the prior for weekly updates.

We immediately rename the important preseason columns so it is impossible to confuse the original ratings with the new weekly ratings.


In [3]:
preseason = pd.read_parquet(
    PRESEASON_STRENGTH_PATH
).copy()

required_preseason_columns = [
    "team",
    "team_strength",
    "baseline_team_strength",
    "personnel_adjustment",
    "personnel_strength",
    "roster_continuity",
    "roster_continuity_adjustment"
]

missing_preseason = [
    column
    for column in required_preseason_columns
    if column not in preseason.columns
]

if missing_preseason:
    raise KeyError(
        "Missing required preseason columns: "
        + ", ".join(missing_preseason)
    )

preseason = preseason.rename(
    columns={
        "strength_rank": "preseason_strength_rank",
        "team_strength": "preseason_team_strength",
        "baseline_team_strength": "preseason_baseline_team_strength",
        "personnel_adjustment": "preseason_personnel_adjustment",
        "personnel_strength": "preseason_personnel_strength",
        "roster_continuity": "preseason_roster_continuity",
        "roster_continuity_adjustment": "preseason_roster_continuity_adjustment"
    }
)

print("Preseason teams:", len(preseason))
print("Unique teams:", preseason["team"].nunique())

display(
    preseason[
        [
            "preseason_strength_rank",
            "team",
            "preseason_team_strength",
            "preseason_baseline_team_strength",
            "preseason_personnel_adjustment",
            "preseason_roster_continuity_adjustment"
        ]
    ].sort_values(
        "preseason_strength_rank"
    )
)


Preseason teams: 32
Unique teams: 32


,preseason_strength_rank,team,preseason_team_strength,preseason_baseline_team_strength,preseason_personnel_adjustment,preseason_roster_continuity_adjustment
0,1,BUF,4.816514,4.125764,0.934506,0.260091
1,2,LA,4.762452,3.257133,2.100811,0.489665
2,3,SEA,4.363421,3.515976,0.635683,0.932073
3,4,DET,4.018565,3.374644,1.041923,0.037533
4,5,DEN,3.569507,2.355154,0.979753,1.253003
5,6,BAL,3.074111,2.710864,0.636254,-0.037012
6,7,HOU,2.810133,2.193179,1.173833,-0.174691
7,8,GB,2.536317,1.718369,0.882628,0.576742
8,9,PHI,2.451919,2.324982,0.264690,-0.063756
9,10,SF,2.366788,1.805709,0.692939,0.290631


# Load Prior-Week Team Strength

The previous week's team strength file is the starting point for the new weekly update.

For Week 3, the model begins with the Week 2 ratings.

This prevents older games from being counted repeatedly and creates a true sequential rating system.

In [4]:
prior_week = pd.read_parquet(
    PRIOR_WEEK_STRENGTH_PATH
).copy()

required_prior_columns = [
    "team",
    "weekly_strength_rank",
    "weekly_team_strength",
    "cumulative_offense_adjustment",
    "cumulative_defense_adjustment"
]

missing_prior = [
    column
    for column in required_prior_columns
    if column not in prior_week.columns
]

if missing_prior:
    raise KeyError(
        "Missing required prior-week columns: "
        + ", ".join(missing_prior)
    )

prior_week = prior_week[
    required_prior_columns
].rename(
    columns={
        "weekly_strength_rank":
            "prior_week_strength_rank",
        "weekly_team_strength":
            "prior_week_team_strength",
        "cumulative_offense_adjustment":
            "prior_cumulative_offense_adjustment",
        "cumulative_defense_adjustment":
            "prior_cumulative_defense_adjustment"
    }
)

print(
    f"Week {PRIOR_WEEK} teams:",
    len(prior_week)
)

print(
    "Unique teams:",
    prior_week["team"].nunique()
)

display(
    prior_week.sort_values(
        "prior_week_strength_rank"
    )
)

Week 8 teams: 32
Unique teams: 32


,team,prior_week_strength_rank,prior_week_team_strength,prior_cumulative_offense_adjustment,prior_cumulative_defense_adjustment
0,SEA,1,5.114660,1.324606,5.672999
1,BUF,2,4.947555,9.052222,-4.620813
2,DEN,3,3.957970,-2.121782,3.426611
3,SF,4,3.931716,5.414943,3.313336
4,LA,5,3.674363,1.325867,0.274260
5,MIN,6,3.147998,-2.364647,9.548032
6,JAX,7,3.094193,2.469038,5.296003
7,NE,8,2.959946,0.174868,3.023261
8,DET,9,2.938408,6.434550,-4.152771
9,KC,10,2.841071,0.858981,4.997374


# Load New Weekly In-Season Features

These features contain the new evidence generated in notebook `03`.

For Week 3, they describe each team's Week 2 performance relative to its entering Week 2 expectation.

The primary values used here are:

- `weekly_strength_update`
- `weekly_offense_adjustment`
- `weekly_defense_adjustment`

The overall strength update is opponent adjusted and uses diminishing returns for extreme single game results.

In [5]:
inseason = pd.read_parquet(
    INSEASON_FEATURES_PATH
).copy()

required_inseason_columns = [
    "team",
    "games_in_update",
    "entering_team_strength",
    "entering_strength_rank",
    "average_opponent_strength",
    "average_opponent_rank",
    "actual_margin",
    "expected_margin",
    "performance_residual",
    "weekly_strength_update",
    "weekly_offense_adjustment",
    "weekly_defense_adjustment"
]

missing_inseason = [
    column
    for column in required_inseason_columns
    if column not in inseason.columns
]

if missing_inseason:
    raise KeyError(
        "Missing required in-season columns: "
        + ", ".join(missing_inseason)
    )

print("In-season teams:", len(inseason))
print(
    "Unique teams:",
    inseason["team"].nunique()
)

display(
    inseason[
        required_inseason_columns
    ].sort_values(
        "weekly_strength_update",
        ascending=False
    )
)

In-season teams: 28
Unique teams: 28


,team,games_in_update,entering_team_strength,entering_strength_rank,average_opponent_strength,average_opponent_rank,actual_margin,expected_margin,performance_residual,weekly_strength_update,weekly_offense_adjustment,weekly_defense_adjustment
13,JAX,1,3.094193,7,-1.300293,20.0,35.0,5.894486,29.105514,1.453803,2.753025,2.399453
5,CHI,1,2.212394,11,5.114660,1.0,17.0,-4.402266,21.402266,1.365311,1.793585,1.140014
3,BUF,1,4.947555,2,0.994015,16.0,26.0,5.453539,20.546461,1.348679,1.939061,2.035490
27,WAS,1,-2.833574,25,-2.100154,21.0,17.0,0.766579,16.233421,1.231330,0.473030,1.919459
23,PIT,1,1.673987,14,-3.181771,26.0,21.0,6.355758,14.644242,1.170380,0.432469,2.478898
25,TB,1,-0.046570,18,-0.500295,19.0,16.0,1.953725,14.046275,1.144468,-1.116975,3.479453
0,ARI,1,-5.245641,31,0.079981,17.0,3.0,-6.825622,9.825622,0.908292,1.454785,-0.998786
15,LA,1,3.674363,5,-2.722905,24.0,15.0,7.897268,7.102732,0.701803,-0.000323,2.046105
17,LV,1,-3.855768,28,-4.607032,29.0,5.0,-0.748736,5.748736,0.583504,2.029022,-1.624549
26,TEN,1,-6.968377,32,2.192357,12.0,-7.0,-10.660734,3.660734,0.383520,1.983999,-2.869572


# Merge Prior Week State with New Evidence

The weekly model now combines three pieces of information:

1. Frozen preseason information for long term reference
2. Previous week's team strength as the current prior
3. Most recent week's opponent adjusted performance as new evidence

All 32 teams must remain in the table.

In [6]:
weekly_strength = (
    preseason
    .merge(
        prior_week,
        on="team",
        how="left",
        validate="one_to_one"
    )
    .merge(
        inseason,
        on="team",
        how="left",
        validate="one_to_one"
    )
)

weekly_fill_columns = [
    "games_in_update",
    "weekly_strength_update",
    "weekly_offense_adjustment",
    "weekly_defense_adjustment"
]

weekly_strength[
    weekly_fill_columns
] = (
    weekly_strength[
        weekly_fill_columns
    ]
    .fillna(0.0)
)

print("Merged teams:", len(weekly_strength))
print(
    "Unique teams:",
    weekly_strength["team"].nunique()
)

print(
    "Missing prior-week strengths:",
    weekly_strength[
        "prior_week_team_strength"
    ].isna().sum()
)

Merged teams: 32
Unique teams: 32
Missing prior-week strengths: 0


# Sequential Weekly Rating Update

The previous week's team strength is the starting point.

The current week's opponent adjusted performance update is then applied:

`weekly_team_strength = prior_week_team_strength + weekly_strength_update`

Therefore, for Week 3:

`Week 3 Strength = Week 2 Strength + Week 2 Performance Update`

Older results remain embedded in the prior week rating and are not counted again.

The offensive and defensive weekly adjustments remain available separately for the weekly scoring model.

In [7]:
weekly_strength["team_strength_change"] = (
    weekly_strength["weekly_strength_update"]
)

weekly_strength["weekly_team_strength"] = (
    weekly_strength["prior_week_team_strength"]
    + weekly_strength["team_strength_change"]
)

display(
    weekly_strength[
        [
            "team",
            "prior_week_team_strength",
            "weekly_offense_adjustment",
            "weekly_defense_adjustment",
            "team_strength_change",
            "weekly_team_strength"
        ]
    ].sort_values(
        "weekly_team_strength",
        ascending=False
    )
)

,team,prior_week_team_strength,weekly_offense_adjustment,weekly_defense_adjustment,team_strength_change,weekly_team_strength
0,BUF,4.947555,1.939061,2.035490,1.348679,6.296234
11,JAX,3.094193,2.753025,2.399453,1.453803,4.547996
1,LA,3.674363,-0.000323,2.046105,0.701803,4.376166
4,DEN,3.957970,0.808326,0.004754,0.147709,4.105679
9,SF,3.931716,0.000000,0.000000,0.000000,3.931716
2,SEA,5.114660,-0.865250,-1.518821,-1.365311,3.749348
18,CHI,2.212394,1.793585,1.140014,1.365311,3.577705
3,DET,2.938408,-0.530164,1.366264,0.286742,3.225150
17,MIN,3.147998,-1.138024,0.758405,-0.286742,2.861256
15,PIT,1.673987,0.432469,2.478898,1.170380,2.844367


In [8]:
weekly_strength["cumulative_offense_adjustment"] = (
    weekly_strength["prior_cumulative_offense_adjustment"]
    + weekly_strength["weekly_offense_adjustment"]
)

weekly_strength["cumulative_defense_adjustment"] = (
    weekly_strength["prior_cumulative_defense_adjustment"]
    + weekly_strength["weekly_defense_adjustment"]
)

assert weekly_strength[
    "cumulative_offense_adjustment"
].notna().all()

assert weekly_strength[
    "cumulative_defense_adjustment"
].notna().all()

# Updated Weekly Rankings

The current weekly rank is based on `weekly_team_strength`.

Ranking movement is now measured relative to the immediately previous week rather than the preseason ranking.

A positive `rank_change` means the team moved up.

For example:

- Week 2 rank = 8
- Week 3 rank = 5
- Rank change = +3

In [9]:
weekly_strength = weekly_strength.sort_values(
    [
        "weekly_team_strength",
        "prior_week_team_strength"
    ],
    ascending=[False, False]
).reset_index(drop=True)

weekly_strength["weekly_strength_rank"] = (
    np.arange(
        1,
        len(weekly_strength) + 1
    )
)

weekly_strength["rank_change"] = (
    weekly_strength["prior_week_strength_rank"]
    - weekly_strength["weekly_strength_rank"]
)

weekly_comparison_table = (
    weekly_strength[
        [
            "weekly_strength_rank",
            "team",
            "weekly_team_strength",
            "prior_week_strength_rank",
            "prior_week_team_strength",
            "team_strength_change",
            "rank_change"
        ]
    ]
    .copy()
)

display(weekly_comparison_table)

,weekly_strength_rank,team,weekly_team_strength,prior_week_strength_rank,prior_week_team_strength,team_strength_change,rank_change
0,1,BUF,6.296234,2,4.947555,1.348679,1
1,2,JAX,4.547996,7,3.094193,1.453803,5
2,3,LA,4.376166,5,3.674363,0.701803,2
3,4,DEN,4.105679,3,3.957970,0.147709,-1
4,5,SF,3.931716,4,3.931716,0.000000,-1
5,6,SEA,3.749348,1,5.114660,-1.365311,-5
6,7,CHI,3.577705,11,2.212394,1.365311,4
7,8,DET,3.225150,9,2.938408,0.286742,1
8,9,MIN,2.861256,6,3.147998,-0.286742,-3
9,10,PIT,2.844367,14,1.673987,1.170380,4


# Biggest Weekly Movers

This table is diagnostic only.

It shows which teams experienced the largest overall rating changes from the previous week to the current week.

The underlying projection model still retains the full preseason, offense, defense, opponent and weekly information.

In [10]:
mover_columns = [
    "team",
    "prior_week_team_strength",
    "team_strength_change",
    "weekly_team_strength",
    "prior_week_strength_rank",
    "weekly_strength_rank",
    "rank_change"
]

biggest_risers = (
    weekly_strength[
        mover_columns
    ]
    .sort_values(
        "team_strength_change",
        ascending=False
    )
    .head(10)
)

biggest_fallers = (
    weekly_strength[
        mover_columns
    ]
    .sort_values(
        "team_strength_change",
        ascending=True
    )
    .head(10)
)

print("BIGGEST RISERS")
display(biggest_risers)

print()
print("BIGGEST FALLERS")
display(biggest_fallers)

BIGGEST RISERS


,team,prior_week_team_strength,team_strength_change,weekly_team_strength,prior_week_strength_rank,weekly_strength_rank,rank_change
1,JAX,3.094193,1.453803,4.547996,7,2,5
6,CHI,2.212394,1.365311,3.577705,11,7,4
0,BUF,4.947555,1.348679,6.296234,2,1,1
18,WAS,-2.833574,1.231330,-1.602244,25,19,6
9,PIT,1.673987,1.170380,2.844367,14,10,4
14,TB,-0.046570,1.144468,1.097898,18,15,3
27,ARI,-5.245641,0.908292,-4.337349,31,28,3
2,LA,3.674363,0.701803,4.376166,5,3,2
23,LV,-3.855768,0.583504,-3.272264,28,24,4
31,TEN,-6.968377,0.383520,-6.584857,32,32,0



BIGGEST FALLERS


,team,prior_week_team_strength,team_strength_change,weekly_team_strength,prior_week_strength_rank,weekly_strength_rank,rank_change
22,IND,-1.300293,-1.453803,-2.754096,20,23,-3
5,SEA,5.114660,-1.365311,3.749348,1,6,-5
16,BAL,0.994015,-1.348679,-0.354664,16,17,-1
24,PHI,-2.100154,-1.231330,-3.331484,21,25,-4
28,CLE,-3.181771,-1.170380,-4.352151,26,29,-3
19,ATL,-0.500295,-1.144468,-1.644763,19,20,-1
17,DAL,0.079981,-0.908292,-0.828311,17,18,-1
25,LAC,-2.722905,-0.701803,-3.424708,24,26,-2
30,NYJ,-4.607032,-0.583504,-5.190536,29,31,-2
13,CIN,2.192357,-0.383520,1.808836,12,14,-2


# Update Diagnostics

These diagnostics check the size and distribution of the sequential weekly update.

The comparison is now:

`Prior Week Strength → Current Week Strength`

rather than:

`Preseason Strength → Current Week Strength`

In [11]:
movement = (
    weekly_strength[
        "team_strength_change"
    ].abs()
)

print(
    f"WEEK {PRIOR_WEEK} → "
    f"WEEK {TARGET_WEEK} TEAM-STRENGTH MOVEMENT"
)

print(
    weekly_strength[
        "team_strength_change"
    ]
    .describe()
    .round(3)
)

print()

print(
    "Mean absolute movement:",
    round(
        movement.mean(),
        3
    )
)

print(
    "Maximum absolute movement:",
    round(
        movement.max(),
        3
    )
)

print(
    "Team with largest movement:",
    weekly_strength.loc[
        movement.idxmax(),
        "team"
    ]
)

print()
print("RATING DISTRIBUTIONS")

distribution_check = pd.DataFrame(
    {
        "metric": [
            f"Week {PRIOR_WEEK} Team Strength",
            f"Week {TARGET_WEEK} Team Strength"
        ],
        "mean": [
            weekly_strength[
                "prior_week_team_strength"
            ].mean(),
            weekly_strength[
                "weekly_team_strength"
            ].mean()
        ],
        "std": [
            weekly_strength[
                "prior_week_team_strength"
            ].std(),
            weekly_strength[
                "weekly_team_strength"
            ].std()
        ],
        "min": [
            weekly_strength[
                "prior_week_team_strength"
            ].min(),
            weekly_strength[
                "weekly_team_strength"
            ].min()
        ],
        "max": [
            weekly_strength[
                "prior_week_team_strength"
            ].max(),
            weekly_strength[
                "weekly_team_strength"
            ].max()
        ]
    }
)

display(
    distribution_check.round(3)
)

WEEK 8 → WEEK 9 TEAM-STRENGTH MOVEMENT
count    32.000
mean      0.000
std       0.877
min      -1.454
25%      -0.613
50%       0.000
75%       0.613
max       1.454
Name: team_strength_change, dtype: float64

Mean absolute movement: 0.683
Maximum absolute movement: 1.454
Team with largest movement: JAX

RATING DISTRIBUTIONS


,metric,mean,std,min,max
0,Week 8 Team Strength,0.002,3.366,-6.968,5.115
1,Week 9 Team Strength,0.002,3.563,-6.585,6.296


# Sanity Checks

These checks verify the sequential weekly update.

The weekly file should:

- contain exactly 32 unique teams
- preserve the frozen preseason information
- contain a valid prior week rating for every team
- contain no missing current weekly strength values
- have `team_strength_change` equal to the new opponent adjusted weekly update
- calculate current strength as prior week strength plus the weekly update
- never overwrite the preseason or prior week files

In [12]:
assert len(weekly_strength) == 32, (
    "Expected 32 teams in weekly strength table."
)

assert (
    weekly_strength["team"].nunique()
    == 32
), (
    "Expected 32 unique teams."
)

assert weekly_strength[
    "preseason_team_strength"
].notna().all(), (
    "Missing frozen preseason ratings."
)

assert weekly_strength[
    "prior_week_team_strength"
].notna().all(), (
    "Missing prior-week ratings."
)

assert weekly_strength[
    "weekly_team_strength"
].notna().all(), (
    "Missing current weekly ratings."
)

assert np.allclose(
    weekly_strength[
        "team_strength_change"
    ],
    weekly_strength[
        "weekly_strength_update"
    ]
), (
    "Team-strength change does not match "
    "the opponent-adjusted weekly update."
)

assert np.allclose(
    weekly_strength[
        "weekly_team_strength"
    ],
    (
        weekly_strength[
            "prior_week_team_strength"
        ]
        + weekly_strength[
            "team_strength_change"
        ]
    )
), (
    "Sequential weekly strength formula "
    "is inconsistent."
)

played_mask = weekly_strength["games_in_update"] > 0

assert np.allclose(
    weekly_strength.loc[
        played_mask, "entering_team_strength"
    ],
    weekly_strength.loc[
        played_mask, "prior_week_team_strength"
    ]
), (
    "Notebook 03 entering strength does not match "
    "the prior-week team strength."
)

bye_mask = ~played_mask

assert np.allclose(
    weekly_strength.loc[
        bye_mask, "weekly_team_strength"
    ],
    weekly_strength.loc[
        bye_mask, "prior_week_team_strength"
    ]
), "Bye-week teams did not retain their previous strength."

print(
    "All sequential weekly team-strength "
    "sanity checks passed."
)

All sequential weekly team-strength sanity checks passed.


# Final Weekly Team Strength Table

The saved file contains the full state needed by the weekly projection pipeline.

This includes:

- current weekly strength and rank
- previous week's strength and rank
- week to week movement
- frozen preseason information
- opponent adjusted performance information
- separate offensive and defensive weekly signals

The compact ranking table shown earlier is only for interpretation. The saved dataset retains the additional model information.

In [13]:
final_columns = [
    "weekly_strength_rank",
    "team",
    "weekly_team_strength",

    "prior_week_strength_rank",
    "prior_week_team_strength",
    "team_strength_change",
    "rank_change",

    "games_in_update",
    "average_opponent_rank",
    "average_opponent_strength",
    "actual_margin",
    "expected_margin",
    "performance_residual",
    "weekly_strength_update",

    "weekly_offense_adjustment",
    "weekly_defense_adjustment",
    "cumulative_offense_adjustment",
    "cumulative_defense_adjustment",

    "preseason_strength_rank",
    "preseason_team_strength",
    "preseason_baseline_team_strength",
    "preseason_personnel_adjustment",
    "preseason_personnel_strength",
    "preseason_roster_continuity",
    "preseason_roster_continuity_adjustment"
]

weekly_team_strength = (
    weekly_strength[
        final_columns
    ]
    .copy()
)

display(
    weekly_team_strength
)

,weekly_strength_rank,team,weekly_team_strength,prior_week_strength_rank,prior_week_team_strength,team_strength_change,rank_change,games_in_update,average_opponent_rank,average_opponent_strength,...,weekly_defense_adjustment,cumulative_offense_adjustment,cumulative_defense_adjustment,preseason_strength_rank,preseason_team_strength,preseason_baseline_team_strength,preseason_personnel_adjustment,preseason_personnel_strength,preseason_roster_continuity,preseason_roster_continuity_adjustment
0,1,BUF,6.296234,2,4.947555,1.348679,1,1.0,16.0,0.994015,...,2.035490,10.991284,-2.585323,1,4.816514,4.125764,0.934506,0.623004,0.573034,0.260091
1,2,JAX,4.547996,7,3.094193,1.453803,5,1.0,20.0,-1.300293,...,2.399453,5.222063,7.695456,12,1.713136,1.746953,-0.223729,-0.149153,0.563830,0.200841
2,3,LA,4.376166,5,3.674363,0.701803,2,1.0,24.0,-2.722905,...,2.046105,1.325544,2.320365,2,4.762452,3.257133,2.100811,1.400540,0.608696,0.489665
3,4,DEN,4.105679,3,3.957970,0.147709,-1,1.0,10.0,2.841071,...,0.004754,-1.313456,3.431365,5,3.569507,2.355154,0.979753,0.653169,0.727273,1.253003
4,5,SF,3.931716,4,3.931716,0.000000,-1,0.0,NaN,NaN,...,0.000000,5.414943,3.313336,10,2.366788,1.805709,0.692939,0.461959,0.577778,0.290631
5,6,SEA,3.749348,1,5.114660,-1.365311,-5,1.0,11.0,2.212394,...,-1.518821,0.459356,4.154178,3,4.363421,3.515976,0.635683,0.423788,0.677419,0.932073
6,7,CHI,3.577705,11,2.212394,1.365311,4,1.0,1.0,5.114660,...,1.140014,4.535805,3.730626,19,-0.051934,-0.069709,0.002783,0.001856,0.537634,0.032209
7,8,DET,3.225150,9,2.938408,0.286742,1,1.0,6.0,3.147998,...,1.366264,5.904386,-2.786507,4,4.018565,3.374644,1.041923,0.694616,0.538462,0.037533
8,9,MIN,2.861256,6,3.147998,-0.286742,-3,1.0,9.0,2.938408,...,0.758405,-3.502671,10.306436,18,0.475040,0.919610,-0.620638,-0.413759,0.510204,-0.144374
9,10,PIT,2.844367,14,1.673987,1.170380,4,1.0,26.0,-3.181771,...,2.478898,-0.929969,5.764852,16,0.543902,0.359049,0.304322,0.202881,0.533333,0.004521


# Save Weekly Team Strength

This writes only to:

`data/processed/weekly/`

The original:

`data/processed/2026_team_strength.parquet`

remains untouched.


In [14]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

weekly_team_strength.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)
print()
print(
    "Original preseason file remains:",
    PRESEASON_STRENGTH_PATH
)


Saved: ..\..\data\processed\weekly\week_09_sequential_team_strength.parquet

Original preseason file remains: ..\..\data\processed\2026_team_strength.parquet


# What This Means for Week 3

At the end of this notebook, every team has:

- its frozen preseason rating
- its Week 2 entering rating and rank
- its opponent adjusted Week 2 performance
- its Week 2 offensive and defensive signals
- its Week 2 → Week 3 rating movement
- its updated Week 3 team strength rating
- its updated Week 3 rank

The next stage will apply the Week 3 injury update and then use the resulting team state to generate Week 3 game projections.

The weekly ranking table is diagnostic only.

The saved weekly team strength file retains the additional information required by the projection pipeline.